# 🔥 AI-CLIP-HUB : Fine-Tuning LLaMA-3 (Unsloth)

Jupyter Notebook ini didesain secara khusus untuk dijalankan di **Google Colab (T4 / L4 GPU)**. 
Kita akan menggunakan pustaka **Unsloth** untuk melatih model 2x lebih cepat dengan VRAM yang jauh lebih hemat.

### Panduan Singkat
1. Buka [Google Colab](https://colab.research.google.com/)
2. Upload notebook ini (`Colab_FineTuning.ipynb`).
3. Pastikan memilih Runtime -> Change Runtime Type -> **T4 GPU**.
4. Upload file `combined_dataset.jsonl` dari folder lokal Anda (`modulTrain/combined_dataset.jsonl`) ke Google Drive Anda.


In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# PASTIKAN PATH INI SESUAI DENGAN LOKASI jsonl ANDA DI GOOGLE DRIVE
dataset_path = "/content/drive/MyDrive/combined_dataset.jsonl"

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Sesuaikan dengan panjang teks instruksi/konten video
dtype = None # Auto deteksi float16/bfloat16
load_in_4bit = True # 4-bit quantization untuk menghemat VRAM Colab T4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Bisa ganti dengan model IndoBERT/Llama 3
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Setting LoRA (Low-Rank Adaptation)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank optimal untuk fine-tuning ringan
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", 
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from datasets import load_dataset

# Format Instruction Prompt menggunakan standard Alpaca
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("json", data_files=dataset_path, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True, remove_columns=dataset.column_names)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        # === HYPERPARAMETERS TUNING KONFIGURASI ===
        per_device_train_batch_size = 4, # Lebih besar = lebih stabil gradients, 4 cukup aman untuk RAM T4.
        gradient_accumulation_steps = 4, # Simulasi global batch size: 4 * 4 = 16 batches global.
        warmup_steps = 10, # Pencegahan loss spike di awal latihan.
        # Menggunakan Epoch dibandingkan Steps terbatas agar seluruh data training lewat secara adil:
        num_train_epochs = 3, # Standar Industri Fine-Tuning Llama adalah 3 putaran (epochs).
        learning_rate = 2e-4, # Angka ajaib untuk adaptasi kognitif Llama-3 LoRA
        # ==========================================
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Optimizer 8-bit sangat efektif untuk VRAM pas-pasan
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
# Eksekusi Training!
trainer_stats = trainer.train()

In [ ]:
# Ujicoba Model setelah ditraining
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Berikan opini provokatif (untuk memancing UGC)", # instruction
        "Topik AI akan mengambil alih dunia dalam 5 tahun.", # input
        "", # output blkng dikosongkan karena model yang akan menyelesaikannya
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

In [ ]:
# Menyimpan model LoRA adapters ke Google Drive
save_path = "/content/drive/MyDrive/aicliphub_lora_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Model berhasil disimpan ke {save_path}")

# Jika ingin export ke format Ollama/GGUF (.gguf) uncomment block di bawah:
# model.save_pretrained_gguf("/content/drive/MyDrive/aicliphub_gguf", tokenizer, quantization_method = "q4_k_m")